In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc

/home/chongxiao/miniconda3/envs/htodemux/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
adata_path = "/old_nfs/chengwang_data/ICML_data/real_data/t-cell-depleted-bm-rna.h5ad"

scvi_path = "/home/chongxiao/projects/InfoGlobe/scvi_results/X_scVI.npy"
scphere_path = "/home/chongxiao/projects/InfoGlobe/methods_src/scPhere/results_scphere2/scphere_latent.tsv"
irvae_path = "/home/chongxiao/projects/InfoGlobe/methods_src/IRVAE-public/results_irvae2/irvae_latent.npy"
flatvi_path = "/home/chongxiao/projects/InfoGlobe/methods_src/FlatVI/project_folder/experiments/flatvi_t_cell/flatvi_latent.npy"

out_h5ad = "/home/chongxiao/projects/InfoGlobe/benchmark_results/t_cell_with_all_embeddings.h5ad"
os.makedirs(os.path.dirname(out_h5ad), exist_ok=True)

In [3]:
adata = sc.read_h5ad(adata_path)
print(adata)
print("existing obsm keys:", list(adata.obsm.keys()))

AnnData object with n_obs × n_vars = 8627 × 17226
    obs: 'sample', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'batch', 'DoubletScores', 'n_counts', 'leiden', 'phenograph', 'log_n_counts', 'celltype', 'palantir_pseudotime', 'selection', 'NaiveB_lineage', 'mellon_log_density', 'mellon_log_density_clipped'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'PeakCounts'
    uns: 'DMEigenValues', 'DM_EigenValues', 'NaiveB_lineage_colors', 'celltype_colors', 'custom_branch_mask_columns', 'hvg', 'leiden', 'mellon_log_density_predictor', 'neighbors', 'pca', 'sample_colors', 'umap'
    obsm: 'DM_EigenVectors', 'X_FDL', 'X_pca', 'X_umap', 'branch_masks', 'chromVAR_deviations', 'palantir_branch_probs', 'palantir_fate_probabilities', 'palantir_lineage_cells'
    varm: 'PCs', 'geneXTF'
    layers: 'Bcells_lineage_specific', 'Bcells_primed', 'MAGIC_imputed_data'
    obsp: 'DM_Kernel', 'DM_Similarity', 'connectivities', 'distances', 'knn

In [4]:
adata.obsm["scvi"] = np.load(scvi_path)

adata.obsm["scphere"] = pd.read_csv(
    scphere_path,
    sep="\t",
    header=None
).values

adata.obsm["irvae"] = np.load(irvae_path)

adata.obsm["flatvi"] = np.load(flatvi_path)

In [5]:
for k in ["scvi", "scphere", "irvae", "flatvi"]:
    print(k, adata.obsm[k].shape)
    assert adata.obsm[k].shape[0] == adata.n_obs, f"{k} rows != n_obs"

scvi (8627, 10)
scphere (8627, 17)
irvae (8627, 16)
flatvi (8627, 10)


In [6]:
adata.write(out_h5ad)
print("saved to:", out_h5ad)
print("final obsm keys:", list(adata.obsm.keys()))

saved to: /home/chongxiao/projects/InfoGlobe/benchmark_results/t_cell_with_all_embeddings.h5ad
final obsm keys: ['DM_EigenVectors', 'X_FDL', 'X_pca', 'X_umap', 'branch_masks', 'chromVAR_deviations', 'palantir_branch_probs', 'palantir_fate_probabilities', 'palantir_lineage_cells', 'scvi', 'scphere', 'irvae', 'flatvi']


In [7]:
adata

AnnData object with n_obs × n_vars = 8627 × 17226
    obs: 'sample', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'batch', 'DoubletScores', 'n_counts', 'leiden', 'phenograph', 'log_n_counts', 'celltype', 'palantir_pseudotime', 'selection', 'NaiveB_lineage', 'mellon_log_density', 'mellon_log_density_clipped'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'PeakCounts'
    uns: 'DMEigenValues', 'DM_EigenValues', 'NaiveB_lineage_colors', 'celltype_colors', 'custom_branch_mask_columns', 'hvg', 'leiden', 'mellon_log_density_predictor', 'neighbors', 'pca', 'sample_colors', 'umap'
    obsm: 'DM_EigenVectors', 'X_FDL', 'X_pca', 'X_umap', 'branch_masks', 'chromVAR_deviations', 'palantir_branch_probs', 'palantir_fate_probabilities', 'palantir_lineage_cells', 'scvi', 'scphere', 'irvae', 'flatvi'
    varm: 'PCs', 'geneXTF'
    layers: 'Bcells_lineage_specific', 'Bcells_primed', 'MAGIC_imputed_data'
    obsp: 'DM_Kernel', 'DM_Similarity